# 📊 InterpScore v0.0.1 — composite SAE evaluation

**One number for an SAE.** InterpScore rolls five of SAEBench's per-metric numbers into a single scalar so two SAEs can be ranked without staring at a scatter plot. The individual components are always reported alongside the composite — if you disagree with our weights, recompute the composite yourself from the components JSON.

### Primary sources

- Karvonen, Bricken, Pearce, et al., *SAEBench: A Comprehensive Benchmark for Sparse Autoencoders*, 2025 — [arXiv:2503.09532](https://arxiv.org/abs/2503.09532). 8-metric framework. **SAEBench explicitly declines to publish a composite score** (see §1 "we do not recommend a single aggregate metric"). InterpScore is a separate, opinionated aggregation built *on top of* SAEBench's raw metrics — the disagreement is intentional and transparent.
- Gao, Goh, Sutskever et al., *Scaling and Evaluating Sparse Autoencoders*, 2024 — [arXiv:2406.04093](https://arxiv.org/abs/2406.04093). Source for `loss_recovered` and L0 conventions.
- Neuronpedia SAEBench dashboard — <https://www.neuronpedia.org/sae-bench>. Scatter-only UI; no composite score. Greenfield for an opinionated scalar.

### Formula (v0.0.1)

All terms live in [0, 1], higher = better.

```
InterpScore = 0.30 * loss_recovered
            + 0.15 * (1 - dead_frac)
            + 0.15 * l0_score            # exp(-|log(L0 / 80)|), peaks at L0≈80
            + 0.25 * sparse_probing_auc
            + 0.15 * tpp_score
```

### Honest disclaimer

- Weights are **v0.0.1** and open to revision — treat them as a starting point, not a verdict.
- Individual components are reported alongside the composite so you can reweight (or drop) any term.
- The L0 sweet spot of 80 is taken from Gao et al. (GPT-2 Small / 2-4B models, residual stream). For very large or very small models, the peak shifts; we will retune in v0.1.
- Sparse probing uses a 2-task subset of SAEBench's default battery to keep T4 runtime ≤ 20 min. Full-battery numbers belong in a paper-grade run.
- Source is open — the weighting, the L0 target, the probing task set, and the TPP concept are all knobs in the config cell.

In [ ]:
!pip install -q -U \
    transformers \
    accelerate \
    safetensors \
    huggingface_hub \
    sae-bench \
    scikit-learn \
    matplotlib \
    tqdm

import torch, transformers, sklearn
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('sklearn', sklearn.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1. Config

Point this at your SAE on HuggingFace. Defaults are Gemma-2-2B L12 — SAEBench's reference configuration — so you can run end-to-end on a free T4 in ≤ 20 min. Swap in Qwen3.6-27B + layer 31 for paper-grade.

In [ ]:
# --- SAE under test ---
HF_SAE_REPO   = 'caiovicentino1/gemma-2-2b-sae-l12-v01'  # <- change me
HF_SAE_FILE   = 'sae.safetensors'                         # filename inside the repo

# --- Base model the SAE was trained on ---
HF_BASE_MODEL = 'google/gemma-2-2b'
LAYER         = 12
D_MODEL       = 2304
D_SAE         = 16384    # dictionary width of your SAE
K             = 64       # TopK sparsity of your SAE (set to None if you use L1 SAEs)
TOKENS_TRAINED = 200_000_000  # reported in the output JSON, not used by the eval

# --- Eval budget ---
EVAL_TOKENS      = 500_000     # held-out eval tokens (loss_recovered, L0, dead_frac)
PROBE_TOKENS     = 100_000     # per probing task
PROBING_TASKS    = ['toxicity', 'sentiment']   # subset of SAEBench defaults
TPP_CONCEPT      = 'toxicity'                   # concept for targeted probe perturbation
TPP_TOP_FEATURES = 20                           # ablate top-k features per concept

# --- Cache (so a dead Colab session doesn't nuke the activation dump) ---
USE_CACHE_DRIVE  = True
CACHE_DIR        = '/content/drive/MyDrive/interpscore_cache'

# --- Misc ---
BATCH_TOKENS    = 4096
SEED            = 0
VERSION         = 'v0.0.1'

import os, math, json, time, random
random.seed(SEED); torch.manual_seed(SEED)

if USE_CACHE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        os.makedirs(CACHE_DIR, exist_ok=True)
        print('cache ->', CACHE_DIR)
    except Exception as e:
        print('drive unavailable, falling back to /tmp:', e)
        CACHE_DIR = '/tmp/interpscore_cache'
        os.makedirs(CACHE_DIR, exist_ok=True)
else:
    CACHE_DIR = '/tmp/interpscore_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

## 2. Auth + load SAE and base model

Base model loads in `bf16` with SDPA attention (no flash-attn — T4 doesn't have it and it's not needed for eval).

In [ ]:
from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()  # interactive fallback

from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map=device,
)
model.eval()

sae_path = hf_hub_download(repo_id=HF_SAE_REPO, filename=HF_SAE_FILE)
sae_sd = load_file(sae_path)
print('SAE keys:', list(sae_sd.keys()))

class TopKSAE(torch.nn.Module):
    """Minimal TopK SAE wrapper — matches SAEBench / Gao et al. convention.
    Expects W_enc (d_model, d_sae), b_enc (d_sae,), W_dec (d_sae, d_model), b_dec (d_model,).
    If your SAE uses different names, rename in the state dict before loading."""
    def __init__(self, sd, d_model, d_sae, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16))
        self.b_enc = torch.nn.Parameter(sd.get('b_enc', torch.zeros(d_sae)).to(torch.bfloat16))
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16))
        self.b_dec = torch.nn.Parameter(sd.get('b_dec', torch.zeros(d_model)).to(torch.bfloat16))
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        if self.k is None:
            return torch.nn.functional.relu(pre)
        vals, idx = pre.topk(self.k, dim=-1)
        vals = torch.nn.functional.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z
    def decode(self, z):
        return z @ self.W_dec + self.b_dec
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

sae = TopKSAE(sae_sd, D_MODEL, D_SAE, K).to(device).eval()
print(f'SAE loaded: d_model={D_MODEL} d_sae={D_SAE} k={K}')

## 3. Core metrics (SAEBench `core_eval`)

`loss_recovered`, `L0`, `dead_frac`, `var_expl`. Formula (Gao 2024 §3):

```
loss_recovered = (L_zero_ablated - L_sae_reconstructed) / (L_zero_ablated - L_clean)
```

`L_clean` is the model's NLL with the original residual; `L_sae_reconstructed` replaces the residual at `LAYER` with the SAE's reconstruction; `L_zero_ablated` replaces it with zeros. A perfect SAE scores 1.0; zero means the SAE is no better than ablating the layer.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

# Held-out stream. pile-10k / c4 validation are both fine; we use c4 for license clarity.
eval_ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)

def stream_token_batches(ds, tok, batch_tokens=BATCH_TOKENS, total_tokens=EVAL_TOKENS):
    buf = []
    seen = 0
    for ex in ds:
        ids = tok(ex['text'], truncation=True, max_length=1024)['input_ids']
        buf.extend(ids)
        while len(buf) >= batch_tokens:
            chunk = buf[:batch_tokens]; buf = buf[batch_tokens:]
            seen += len(chunk)
            yield torch.tensor(chunk, device=device).unsqueeze(0)
            if seen >= total_tokens:
                return

# --- Hooked forward: capture residual at LAYER, optionally replace ---
_captured = {}
def capture_hook(mod, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    _captured['resid'] = h
    return out

def replace_hook_factory(new_resid_fn):
    def hook(mod, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        new_h = new_resid_fn(h)
        if isinstance(out, tuple):
            return (new_h,) + out[1:]
        return new_h
    return hook

def get_layer_module(model, layer):
    # gemma-2: model.model.layers[layer]; universal fallback walks .layers
    base = getattr(model, 'model', model)
    layers = getattr(base, 'layers', None) or getattr(getattr(base, 'language_model', base), 'layers')
    return layers[layer]

layer_mod = get_layer_module(model, LAYER)

def nll(input_ids, mode='clean'):
    """mode in {clean, sae, zero}."""
    handles = []
    if mode == 'clean':
        pass
    elif mode == 'sae':
        def new_resid(h):
            flat = h.reshape(-1, h.shape[-1])
            recon, _ = sae(flat)
            return recon.reshape(h.shape).to(h.dtype)
        handles.append(layer_mod.register_forward_hook(replace_hook_factory(new_resid)))
    elif mode == 'zero':
        handles.append(layer_mod.register_forward_hook(replace_hook_factory(lambda h: torch.zeros_like(h))))
    with torch.no_grad():
        out = model(input_ids, labels=input_ids)
    for h in handles:
        h.remove()
    return out.loss.item()

# --- Pass 1: accumulate losses + feature activity ---
l_clean_sum = l_sae_sum = l_zero_sum = 0.0
n_batches = 0
feature_active = torch.zeros(D_SAE, dtype=torch.long, device=device)
l0_sum = 0.0
tok_count = 0

# activation cache for probing reuse
act_cache_path = os.path.join(CACHE_DIR, f'acts_{LAYER}.pt')
acts_for_probing = []

h_cap = layer_mod.register_forward_hook(capture_hook)
for batch in tqdm(stream_token_batches(eval_ds, tok), desc='core eval'):
    l_clean_sum += nll(batch, 'clean')
    # re-capture for SAE stats (the zero/sae modes use hooks that replace output)
    with torch.no_grad():
        _ = model(batch)
    resid = _captured['resid']
    flat = resid.reshape(-1, resid.shape[-1])
    with torch.no_grad():
        z = sae.encode(flat)
    feature_active |= (z.abs() > 0).any(dim=0).long()
    l0_sum += (z != 0).float().sum(dim=-1).mean().item()
    tok_count += flat.shape[0]
    if len(acts_for_probing) * BATCH_TOKENS < PROBE_TOKENS * len(PROBING_TASKS):
        acts_for_probing.append(flat.float().cpu())
    # we deliberately recompute nll for sae/zero modes on same batch
    l_sae_sum  += nll(batch, 'sae')
    l_zero_sum += nll(batch, 'zero')
    n_batches += 1
h_cap.remove()

L_clean = l_clean_sum / n_batches
L_sae   = l_sae_sum   / n_batches
L_zero  = l_zero_sum  / n_batches
loss_recovered = (L_zero - L_sae) / max(L_zero - L_clean, 1e-8)
loss_recovered = float(max(0.0, min(1.0, loss_recovered)))

l0 = l0_sum / n_batches
dead_frac = 1.0 - (feature_active.sum().item() / D_SAE)

torch.save(torch.cat(acts_for_probing, dim=0), act_cache_path)
print(f'L_clean={L_clean:.4f}  L_sae={L_sae:.4f}  L_zero={L_zero:.4f}')
print(f'loss_recovered={loss_recovered:.4f}  L0={l0:.2f}  dead_frac={dead_frac:.4f}')

## 4. Sparse probing

For each labelled concept in `PROBING_TASKS`, we collect SAE feature activations on the concept-bearing text, fit a sklearn `LogisticRegression`, and record per-class AUROC. Final `sparse_probing_auc` = mean over tasks. SAEBench ships a 10-task battery; we default to 2 for T4 budget.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Minimal labelled sets. Paper-grade swaps in SAEBench's curated splits.
PROBING_SOURCES = {
    'toxicity': ('jigsaw_toxicity_pred', None, 'comment_text', 'toxic'),
    'sentiment': ('sst2', None, 'sentence', 'label'),
}

def load_labelled(task, n=2000):
    """Return list of (text, label int in {0,1})."""
    try:
        name, cfg, tcol, lcol = PROBING_SOURCES[task]
        ds = load_dataset(name, cfg, split='train', streaming=True) if cfg else load_dataset(name, split='train', streaming=True)
        out = []
        for ex in ds:
            t = ex.get(tcol); l = ex.get(lcol)
            if t is None or l is None: continue
            out.append((t, int(l > 0) if isinstance(l, float) else int(bool(l))))
            if len(out) >= n: break
        return out
    except Exception as e:
        print(f'[warn] could not load {task}, falling back to synthetic split:', e)
        # synthetic fallback: split C4 by length parity so the probe is non-trivial but the eval still runs
        ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
        out = []
        for ex in ds:
            out.append((ex['text'][:512], len(ex['text']) % 2))
            if len(out) >= n: break
        return out

def featurise(texts):
    feats = []
    for t in texts:
        ids = tok(t, truncation=True, max_length=256, return_tensors='pt')['input_ids'].to(device)
        with torch.no_grad():
            _captured.clear()
            h = layer_mod.register_forward_hook(capture_hook)
            _ = model(ids)
            h.remove()
            resid = _captured['resid']           # (1, T, D)
            flat = resid.reshape(-1, resid.shape[-1])
            z = sae.encode(flat).float().cpu()   # (T, N)
            feats.append(z.mean(dim=0).numpy())  # mean-pool over tokens
    import numpy as np
    return np.stack(feats, axis=0)

per_task_auc = {}
for task in PROBING_TASKS:
    data = load_labelled(task, n=1500)
    import numpy as np
    random.Random(SEED).shuffle(data)
    split = int(0.8 * len(data))
    tr, te = data[:split], data[split:]
    X_tr = featurise([x for x,_ in tr]); y_tr = np.array([y for _,y in tr])
    X_te = featurise([x for x,_ in te]); y_te = np.array([y for _,y in te])
    clf = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced').fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    per_task_auc[task] = float(auc)
    print(f'  {task:12s}  AUROC={auc:.4f}  (n_train={len(tr)}, n_test={len(te)})')

sparse_probing_auc = float(sum(per_task_auc.values()) / len(per_task_auc))
print(f'sparse_probing_auc (mean) = {sparse_probing_auc:.4f}')

## 5. TPP — Targeted Probe Perturbation

Causal faithfulness check (SAEBench §4.4). For a concept probe (here `TPP_CONCEPT`), we find the top-k SAE features by probe weight, ablate them in the residual, and measure the AUROC drop. Larger drop = features were genuinely causal for the concept.

```
tpp_score = clip( (AUROC_clean - AUROC_ablated) / AUROC_clean , 0, 1 )
```

In [ ]:
import numpy as np

task = TPP_CONCEPT
data = load_labelled(task, n=1000)
random.Random(SEED).shuffle(data)
split = int(0.8 * len(data))
tr, te = data[:split], data[split:]

X_tr = featurise([x for x,_ in tr]); y_tr = np.array([y for _,y in tr])
X_te = featurise([x for x,_ in te]); y_te = np.array([y for _,y in te])
clf = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced').fit(X_tr, y_tr)
auc_clean = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])

# Pick top features by |probe weight|
w = clf.coef_[0]
top_feats = np.argsort(-np.abs(w))[:TPP_TOP_FEATURES].tolist()
print(f'ablating top-{TPP_TOP_FEATURES} features for {task}: {top_feats[:10]}...')

# Featurise te again with those features zeroed out in the SAE latents
def featurise_ablated(texts, ablate_idx):
    feats = []
    mask = torch.ones(D_SAE, device=device, dtype=torch.bfloat16)
    mask[ablate_idx] = 0.0
    for t in texts:
        ids = tok(t, truncation=True, max_length=256, return_tensors='pt')['input_ids'].to(device)
        with torch.no_grad():
            _captured.clear()
            h = layer_mod.register_forward_hook(capture_hook)
            _ = model(ids)
            h.remove()
            resid = _captured['resid']
            flat = resid.reshape(-1, resid.shape[-1])
            z = sae.encode(flat) * mask
            feats.append(z.float().cpu().mean(dim=0).numpy())
    return np.stack(feats, axis=0)

X_te_ab = featurise_ablated([x for x,_ in te], top_feats)
auc_abl = roc_auc_score(y_te, clf.predict_proba(X_te_ab)[:, 1])

tpp_raw = (auc_clean - auc_abl) / max(auc_clean, 1e-8)
tpp_score = float(max(0.0, min(1.0, tpp_raw)))
print(f'AUROC clean={auc_clean:.4f}  ablated={auc_abl:.4f}  TPP={tpp_score:.4f}')

## 6. Compute InterpScore

In [ ]:
def l0_score(L0, target=80):
    """Peaks at L0 == target, decays exponentially in log space on both sides."""
    if L0 <= 0:
        return 0.0
    return math.exp(-abs(math.log(L0 / target)))

components = {
    'loss_recovered': float(loss_recovered),
    'alive':          float(1 - dead_frac),
    'l0_score':       float(l0_score(l0)),
    'sparse_probing': float(sparse_probing_auc),
    'tpp':            float(tpp_score),
}
weights = {
    'loss_recovered': 0.30,
    'alive':          0.15,
    'l0_score':       0.15,
    'sparse_probing': 0.25,
    'tpp':            0.15,
}
assert abs(sum(weights.values()) - 1.0) < 1e-9

interp_score = float(sum(components[k] * weights[k] for k in weights))

print('components:')
for k, v in components.items():
    print(f'  {k:16s} = {v:.4f}  (w={weights[k]:.2f})')
print(f'\nInterpScore {VERSION} = {interp_score:.4f}')

## 7. Submit to the leaderboard

We write `interpscore.json` + `interpscore_chart.png` into the SAE repo. Two submission paths:

1. **Automated (Q2 2026)** — POST the JSON to <https://openinterp.org/interpscore/submit>. Endpoint is stubbed; the route will land with the dashboard migration.
2. **Manual (today)** — open a PR against [`OpenInterpretability/web`](https://github.com/OpenInterpretability/web) adding your HF repo to `lib/leaderboard.ts`. The site's nightly cron will pull the `interpscore.json` you just uploaded.

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone

payload = {
    'version':        VERSION,
    'sae_repo':       HF_SAE_REPO,
    'model':          HF_BASE_MODEL,
    'layer':          LAYER,
    'd_model':        D_MODEL,
    'd_sae':          D_SAE,
    'k':              K,
    'tokens_trained': TOKENS_TRAINED,
    'eval_tokens':    EVAL_TOKENS,
    'probing_tasks':  PROBING_TASKS,
    'probing_per_task_auc': per_task_auc,
    'tpp_concept':    TPP_CONCEPT,
    'tpp_top_features': TPP_TOP_FEATURES,
    'components':     components,
    'weights':        weights,
    'interp_score':   interp_score,
    'timestamp':      datetime.now(timezone.utc).isoformat(),
    'sources': {
        'saebench': 'arxiv:2503.09532',
        'gao2024':  'arxiv:2406.04093',
        'neuronpedia': 'https://www.neuronpedia.org/sae-bench',
    },
}
out_path = os.path.join(CACHE_DIR, 'interpscore.json')
with open(out_path, 'w') as f:
    json.dump(payload, f, indent=2)
print('wrote', out_path)

api = HfApi()
api.upload_file(
    path_or_fileobj=out_path,
    path_in_repo='interpscore.json',
    repo_id=HF_SAE_REPO,
    repo_type='model',
    commit_message=f'InterpScore {VERSION}: {interp_score:.4f}',
)
print(f'uploaded interpscore.json -> https://huggingface.co/{HF_SAE_REPO}/blob/main/interpscore.json')

print('\n--- Submission ---')
print(f'  [Q2 2026] POST {json.dumps(payload)[:80]}... to https://openinterp.org/interpscore/submit')
print(f'  [Today ] PR to OpenInterpretability/web adding `{HF_SAE_REPO}` to lib/leaderboard.ts')

## 8. Chart

Bar chart of the 5 components (with their weights overlaid) + a pie showing each component's contribution to the composite.

In [ ]:
import matplotlib.pyplot as plt

keys = list(components.keys())
vals = [components[k] for k in keys]
ws   = [weights[k]   for k in keys]
contribs = [components[k] * weights[k] for k in keys]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

bars = ax1.bar(keys, vals, color='#4C72B0')
for bar, w in zip(bars, ws):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'w={w:.2f}', ha='center', fontsize=9, color='#555')
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('component value (∈ [0,1])')
ax1.set_title(f'InterpScore {VERSION} components — composite = {interp_score:.4f}')
ax1.tick_params(axis='x', rotation=20)

ax2.pie(contribs, labels=[f'{k}\n{c:.3f}' for k, c in zip(keys, contribs)],
        autopct='%1.0f%%', startangle=90,
        colors=['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974'])
ax2.set_title('contribution to composite')

plt.tight_layout()
chart_path = os.path.join(CACHE_DIR, 'interpscore_chart.png')
plt.savefig(chart_path, dpi=140, bbox_inches='tight')
plt.show()
print('saved', chart_path)

api.upload_file(
    path_or_fileobj=chart_path,
    path_in_repo='interpscore_chart.png',
    repo_id=HF_SAE_REPO,
    repo_type='model',
    commit_message=f'InterpScore {VERSION} chart',
)
print(f'uploaded chart -> https://huggingface.co/{HF_SAE_REPO}/blob/main/interpscore_chart.png')